# SEVIR: tormentas

Preprocesamiento del experimento de tormentas con SEVIR (Storm EVent ImagRy, MIT Lincoln Lab; Veillette et al., 2020).

SEVIR guarda ~20 000 eventos de 384 km × 384 km × 4 h (49 imágenes cada 5 min) sobre Estados Unidos continental
(2017–2019). Cada evento puede tener 5 modalidades alineadas:

| Modalidad | Contenido | Forma |
|---|---|---|
| `vil` | radar, líquido integrado vertical | 384×384×49 uint8 (1 km/px) |
| `ir069` | GOES-16 canal 9 (vapor de agua) | 192×192×49 int16 (2 km/px) |
| `ir107` | GOES-16 canal 13 (IR ventana) | 192×192×49 int16 (2 km/px) |
| `vis` | GOES-16 canal 2 (visible) | 768×768×49 int16 (0.5 km/px) |
| `lght` | rayos GLM, lista (N, 5) [t s, lat, lon, x px, y px] sobre la malla de 48×48 | |

El experimento solo usa `vil`.

1. **Descarga**: eventos sueltos del bucket público de AWS (`s3://sevir`, sin credenciales) a `Dataset/SEVIR/`.
2. **Estados**: VIL → evoluciones de 7 estados en 96×96 en `CA/SEVIR/<conjunto>/`.

In [ ]:
import csv
import json
import sys
import urllib.request
from collections import defaultdict
from pathlib import Path

import numpy as np

TRAINING = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "scripts" / "main.py").exists())
DATASET = TRAINING / "Dataset" / "SEVIR"
EVENTOS = DATASET / "events"
BASE_URL = "https://sevir.s3.us-west-2.amazonaws.com/"
TYPES = ["vil", "ir069", "ir107", "vis", "lght"]

## 1. Descarga

Los `.h5` originales pesan 4–16 GB y agrupan cientos de eventos, pero cada evento es un bloque contiguo, así que se lee
solo ese bloque por HTTP (rango de bytes) sin bajar el archivo.

- `n = 0` toma todos los candidatos; por default se toma un evento por día (`all_days=True` permite varios).
- `mexico=True` elige los eventos cuyo recuadro cruza la frontera con México (mayor fracción al sur primero). SEVIR no
  cubre México: esa parte sale solo hasta donde llega el radar NEXRAD (~200–250 km dentro de México).
- `south_first=True` ordena por la latitud del centro, del sur al norte.
- `shard="i/N"` toma uno de cada N eventos, para repartir la descarga entre N copias del notebook.

Escribe `Dataset/SEVIR/CATALOG.csv` (34 MB, una vez), `Dataset/SEVIR/events/<id>.npz` (un arreglo por modalidad,
(H, W, 49) con la fila 0 en el sur, más `meta` con la fila del catálogo en JSON) y, con `preview=True`, `<id>.png`.

In [ ]:
# Frontera EUA–México aproximada (lon, lat), de Tijuana a Matamoros siguiendo el río Bravo.
BORDER = [(-117.1, 32.5), (-114.7, 32.7), (-111.1, 31.3), (-108.2, 31.3), (-108.2, 31.8),
          (-106.5, 31.8), (-104.9, 30.6), (-104.4, 29.6), (-103.2, 29.0), (-102.4, 29.8),
          (-101.4, 29.8), (-100.3, 28.3), (-99.5, 27.5), (-99.1, 26.5), (-97.1, 25.9)]


def mexico_fraction(row, samples=24):
    """Fracción aproximada del recuadro (en lat/lon) al sur de la frontera, entre Tijuana y Matamoros."""
    lon = np.linspace(float(row["llcrnrlon"]), float(row["urcrnrlon"]), samples)
    lat = np.linspace(float(row["llcrnrlat"]), float(row["urcrnrlat"]), samples)
    blon, blat = zip(*BORDER)
    border_lat = np.interp(lon, blon, blat)
    inside = (lat[:, None] < border_lat[None, :]) & ((lon >= blon[0]) & (lon <= blon[-1]))[None, :]
    return inside.mean()


def load_catalog(out):
    path = out / "CATALOG.csv"
    if not path.exists():
        print("Descargando CATALOG.csv …")
        urllib.request.urlretrieve(BASE_URL + "CATALOG.csv", path)
    events = defaultdict(dict)
    with open(path) as f:
        for row in csv.DictReader(f):
            events[row["id"]][row["img_type"]] = row
    return events


def read_event(event_id, row, cache):
    """Lee la modalidad de un evento con lecturas parciales del .h5 remoto."""
    import fsspec
    import h5py

    name = row["file_name"]
    if name not in cache:
        cache[name] = h5py.File(fsspec.open(BASE_URL + "data/" + name, block_size=2**20).open(), "r")
    h5 = cache[name]
    if row["img_type"] == "lght":
        return h5[event_id][:] if event_id in h5 else np.zeros((0, 5), np.float32)
    index = int(row["file_index"])
    assert h5["id"][index].decode() == event_id, f"{event_id}: el índice del catálogo no coincide"
    return h5[row["img_type"]][index]


def save_preview(path, event_id, arrays, meta):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    frames = [0, 10, 20, 30, 40, 48]
    images = [t for t in TYPES if t in arrays and t != "lght"]
    fig, axes = plt.subplots(len(images), len(frames), figsize=(2.2 * len(frames), 2.3 * len(images)),
                             squeeze=False)
    cmaps = {"vil": "turbo", "ir069": "viridis_r", "ir107": "gray_r", "vis": "gray"}
    for r, t in enumerate(images):
        data = arrays[t]
        vmin, vmax = np.percentile(data, [1, 99.5])
        for c, k in enumerate(frames):
            ax = axes[r, c]
            ax.imshow(data[:, :, k], cmap=cmaps[t], vmin=vmin, vmax=vmax, origin="lower")  # fila 0 = sur
            if t == "vil" and "lght" in arrays and len(arrays["lght"]):
                # rayos en [t_k, t_k + 5 min) como en el tutorial de SEVIR (t_k = 5k − 120 min),
                # de la malla 48×48 (fila = y desde el sur, columna = x) a la de 384×384
                l = arrays["lght"]
                t0 = (5 * k - 120) * 60
                sel = (l[:, 0] >= t0) & (l[:, 0] < t0 + 300)
                ax.scatter(l[sel, 3] * 8 + 4, l[sel, 4] * 8 + 4, s=1, c="w", lw=0)
                ax.set_xlim(-0.5, data.shape[1] - 0.5); ax.set_ylim(-0.5, data.shape[0] - 0.5)
            ax.set_xticks([]); ax.set_yticks([])
            if r == 0:
                ax.set_title(f"frame {k}", fontsize=9)
            if c == 0:
                ax.set_ylabel(t, fontsize=10)
    first = next(iter(meta.values()))
    fig.suptitle(f"{event_id}  {first['event_type'] or 'random'}  {first['time_utc']} UTC  "
                 f"({float(first['llcrnrlat']):.1f}, {float(first['llcrnrlon']):.1f}) – "
                 f"({float(first['urcrnrlat']):.1f}, {float(first['urcrnrlon']):.1f})", fontsize=10)
    fig.tight_layout()
    fig.savefig(path, dpi=110)
    plt.close(fig)


def descargar(ids=None, n=3, types=TYPES, mexico=False, min_frac=0.15, storms=False, south_first=False,
              all_days=False, skip_existing=False, shard="0/1", preview=True, out=DATASET):
    out = Path(out)
    (out / "events").mkdir(parents=True, exist_ok=True)
    catalog = load_catalog(out)

    if ids:
        missing = [i for i in ids if i not in catalog]
        if missing:
            sys.exit(f"No están en el catálogo: {' '.join(missing)}")
        chosen = list(ids)
    else:
        # eventos con todas las modalidades pedidas y sin datos faltantes
        candidates = [i for i, e in catalog.items()
                      if all(t in e for t in types)
                      and all(float(e[t]["pct_missing"] or 0) == 0 for t in types)
                      and (not storms or i.startswith("S"))]
        if mexico:
            frac = {i: mexico_fraction(next(iter(catalog[i].values()))) for i in candidates}
            candidates = sorted((i for i in candidates if frac[i] >= min_frac), key=lambda i: -frac[i])
        elif south_first:
            def center_lat(i):
                row = next(iter(catalog[i].values()))
                return (float(row["llcrnrlat"]) + float(row["urcrnrlat"])) / 2
            candidates.sort(key=center_lat)
        # un evento por día para no repetir la misma tormenta, salvo con all_days
        chosen, days = [], set()
        for i in candidates:
            day = next(iter(catalog[i].values()))["time_utc"][:10]
            if all_days or day not in days:
                chosen.append(i)
                days.add(day)
            if len(chosen) == n:
                break
        print(f"{len(candidates)} eventos candidatos, se eligen {len(chosen)}")

    shard, shards = map(int, shard.split("/"))
    chosen = chosen[shard::shards]
    if skip_existing:
        chosen = [i for i in chosen if not (out / "events" / f"{i}.npz").exists()]
    print(f"Se descargan {len(chosen)}")

    cache, failed = {}, []
    for event_id in chosen:
        meta = {t: catalog[event_id][t] for t in types if t in catalog[event_id]}
        try:
            arrays = {t: read_event(event_id, row, cache) for t, row in meta.items()}
        except Exception as e:  # red caída o bloque ilegible: se sigue con el siguiente evento
            print(f"{event_id}  FALLÓ: {e!r}", flush=True)
            failed.append(event_id)
            cache.clear()
            continue
        path = out / "events" / f"{event_id}.npz"
        # se escribe a un temporal y se renombra para que skip_existing no tome archivos a medias
        tmp = path.with_suffix(".part.npz")
        np.savez_compressed(tmp, meta=json.dumps(meta), **arrays)
        tmp.rename(path)
        first = next(iter(meta.values()))
        shapes = "  ".join(f"{t} {'×'.join(map(str, a.shape))}" for t, a in arrays.items())
        extra = f"  México ≈{mexico_fraction(first):.0%}" if mexico else ""
        print(f"{event_id}  {first['time_utc']}  {first['event_type'] or 'random':<17} {shapes}{extra}"
              f"  → {path} ({path.stat().st_size / 2**20:.1f} MB)", flush=True)
        if preview:
            save_preview(path.with_suffix(".png"), event_id, arrays, meta)
    if failed:
        print(f"Fallaron {len(failed)}: {' '.join(failed)}")

Todas las tormentas (eventos tipo S), solo VIL, que es lo que necesita el experimento. Si se corta, volver a
correr la celda (salta los ya descargados). Para una prueba rápida: `descargar(mexico=True, n=3)`.

In [ ]:
descargar(types=["vil"], storms=True, south_first=True, n=0, all_days=True, skip_existing=True, preview=False)

## 2. Estados

1. Reduce cada imagen de VIL a (384 / Z)² células promediando bloques de Z×Z píxeles (Z = `zoom` = 4 → 96×96, 4 km).
2. Asigna 7 estados con los umbrales de VIL del artículo de SEVIR: 16, 74, 133, 160, 181 y 219 (0 = sin lluvia,
   6 = la más intensa). Tormenta = estado ≥ 2.
3. Elige al azar (semilla `seed`) `n` tormentas de `year` con al menos `min_frac` de células en estado ≥ 2 en la
   primera imagen (que haya tormenta desde el inicio); `n = 0` = todas.
4. Escribe `CA/SEVIR/<conjunto>/evo.npy` (N, 49, 96, 96) int8 y `evo.json` (eventos y umbrales).

In [ ]:
UMBRALES = [16, 74, 133, 160, 181, 219]


def estados(vil: np.ndarray, zoom: int) -> np.ndarray:
    """vil (H, W, T) uint8 → (T, H/zoom, W/zoom) int8."""
    H, W, T = vil.shape
    m = vil.astype(np.float32).reshape(H // zoom, zoom, W // zoom, zoom, T).mean((1, 3))
    return np.digitize(m, UMBRALES).astype(np.int8).transpose(2, 0, 1)


def evoluciones(year, n, out, zoom=4, min_frac=0.02, seed=0):
    candidatos = []
    for f in sorted(EVENTOS.glob("S*.npz")):
        d = np.load(f)
        meta = json.loads(str(d["meta"]))["vil"]
        if not meta["time_utc"].startswith(str(year)):
            continue
        candidatos.append((f, meta))
    rng = np.random.default_rng(seed)
    rng.shuffle(candidatos)

    evo, elegidos = [], []
    for f, meta in candidatos:
        S = estados(np.load(f)["vil"], zoom)
        if (S[0] >= 2).mean() < min_frac:
            continue
        evo.append(S)
        elegidos.append({"id": meta["id"], "time_utc": meta["time_utc"], "event_type": meta["event_type"],
                         "frac_estado2_inicio": round(float((S[0] >= 2).mean()), 4)})
        if len(evo) == n:
            break
    evo = np.stack(evo)
    out = Path(out)
    out.mkdir(parents=True, exist_ok=True)
    np.save(out / "evo.npy", evo)
    (out / "evo.json").write_text(json.dumps({
        "simulation": "sevir_vil", "year": year, "zoom": zoom, "cell_km": zoom, "step_min": 5,
        "thresholds_vil": UMBRALES, "k": len(UMBRALES) + 1, "min_frac": min_frac, "seed": seed,
        "candidates": len(candidatos), "events": elegidos}, indent=1, ensure_ascii=False))
    print(f"{year}: {len(candidatos)} tormentas, {len(evo)} elegidas → {out / 'evo.npy'} {evo.shape}; "
          f"proporción por estado {np.round(np.bincount(evo.ravel(), minlength=7) / evo.size, 4)}")

Los conjuntos del reporte: el primer experimento entrena con `entrenamiento` y prueba con `prueba`; el barrido entrena
con `todo_2017` + `todo_2018` y prueba con `prueba_200`.

In [ ]:
CA = TRAINING / "CA" / "SEVIR"
evoluciones(2018, 40, CA / "entrenamiento")
evoluciones(2019, 20, CA / "prueba")
evoluciones(2019, 200, CA / "prueba_200")
evoluciones(2017, 0, CA / "todo_2017")
evoluciones(2018, 0, CA / "todo_2018")